In [1]:
#!/usr/bin/env python3
"""
Network Speed Diagnostic & Optimizer
=====================================
Diagnoses slow download speed and applies fixes automatically.
Author: For Eslam - Network Troubleshooting Tool
"""

import os
import sys
import time
import subprocess
import platform
import socket
import struct

# ─────────────────────────────────────────────
# Auto-install required packages
# ─────────────────────────────────────────────
def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

required = ["psutil", "speedtest-cli", "requests", "colorama"]
for pkg in required:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        print(f"[*] Installing {pkg}...")
        install(pkg)

import psutil
import requests
import speedtest
from colorama import Fore, Style, init

init(autoreset=True)

OS = platform.system()  # Windows / Linux / Darwin

# ─────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────
def header(title):
    print(f"\n{Fore.CYAN}{'═'*55}")
    print(f"  {Fore.YELLOW}{title}")
    print(f"{Fore.CYAN}{'═'*55}{Style.RESET_ALL}")

def ok(msg):    print(f"  {Fore.GREEN}[✓] {msg}")
def warn(msg):  print(f"  {Fore.YELLOW}[!] {msg}")
def info(msg):  print(f"  {Fore.CYAN}[i] {msg}")
def bad(msg):   print(f"  {Fore.RED}[✗] {msg}")
def tip(msg):   print(f"  {Fore.MAGENTA}[→] {msg}")

def mb(bytes_val):
    return round(bytes_val / (1024 * 1024), 2)

def mbps(bits_per_sec):
    return round(bits_per_sec / 1_000_000, 2)

# ─────────────────────────────────────────────
# 1. SPEED TEST
# ─────────────────────────────────────────────
def run_speed_test():
    header("📡 Speed Test (Download / Upload / Ping)")
    print("  Running speed test... (this takes ~30 seconds)\n")
    try:
        st = speedtest.Speedtest()
        st.get_best_server()
        download = st.download()
        upload = st.upload()
        ping = st.results.ping
        server = st.results.server

        d_mbps = mbps(download)
        u_mbps = mbps(upload)

        print(f"  Server  : {server['sponsor']} ({server['country']})")
        print(f"  Ping    : {Fore.YELLOW}{ping:.1f} ms")
        print(f"  Download: {Fore.GREEN}{d_mbps} Mbps")
        print(f"  Upload  : {Fore.GREEN}{u_mbps} Mbps")

        if d_mbps < 5:
            bad(f"Download speed is critically low ({d_mbps} Mbps)!")
            tip("Check if ISP is throttling you or if the modem has an issue.")
        elif d_mbps < 20:
            warn(f"Download speed is below average ({d_mbps} Mbps).")
            tip("Possible congestion, weak signal, or router issue.")
        else:
            ok(f"Download speed looks good ({d_mbps} Mbps).")

        if ping > 100:
            warn(f"High ping ({ping:.1f} ms) — causes lag/buffering.")
        else:
            ok(f"Ping is acceptable ({ping:.1f} ms).")

        return d_mbps, u_mbps, ping

    except Exception as e:
        bad(f"Speed test failed: {e}")
        return None, None, None

# ─────────────────────────────────────────────
# 2. ACTIVE NETWORK INTERFACES
# ─────────────────────────────────────────────
def check_interfaces():
    header("🔌 Network Interfaces")
    stats = psutil.net_if_stats()
    addrs = psutil.net_if_addrs()

    for iface, stat in stats.items():
        if not stat.isup:
            continue
        speed_label = f"{stat.speed} Mbps" if stat.speed > 0 else "Unknown"
        addr_list = [a.address for a in addrs.get(iface, [])
                     if a.family == socket.AF_INET]
        ip = addr_list[0] if addr_list else "No IP"

        if "wi" in iface.lower() or "wlan" in iface.lower() or "wlp" in iface.lower():
            icon = "📶"
        elif "eth" in iface.lower() or "en" in iface.lower():
            icon = "🔗"
        else:
            icon = "🌐"

        print(f"  {icon} {Fore.WHITE}{iface:<15} IP: {Fore.CYAN}{ip:<18} "
              f"Link: {Fore.GREEN}{speed_label}")

# ─────────────────────────────────────────────
# 3. BANDWIDTH HOG DETECTION
# ─────────────────────────────────────────────
def find_bandwidth_hogs():
    header("🐷 Bandwidth Hog Processes (Top 10)")
    print("  Sampling for 3 seconds...")

    snapshot1 = {}
    for proc in psutil.process_iter(['pid', 'name']):
        try:
            conns = proc.net_connections()
            snapshot1[proc.pid] = (proc.name(), len(conns))
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass

    net_before = psutil.net_io_counters(pernic=False)
    time.sleep(3)
    net_after = psutil.net_io_counters(pernic=False)

    total_dl = mb(net_after.bytes_recv - net_before.bytes_recv)
    total_ul = mb(net_after.bytes_sent - net_before.bytes_sent)

    print(f"\n  Total in last 3 sec → ↓ {total_dl} MB  |  ↑ {total_ul} MB\n")

    # Get processes with active connections
    hogs = []
    for proc in psutil.process_iter(['pid', 'name', 'status']):
        try:
            conns = proc.net_connections()
            if conns:
                hogs.append((len(conns), proc.pid, proc.name()))
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass

    hogs.sort(reverse=True)
    print(f"  {'Process':<30} {'PID':<8} {'Open Connections'}")
    print(f"  {'─'*50}")
    for count, pid, name in hogs[:10]:
        color = Fore.RED if count > 10 else Fore.YELLOW if count > 5 else Fore.WHITE
        print(f"  {color}{name:<30} {pid:<8} {count}")

    if hogs and hogs[0][0] > 15:
        warn(f"'{hogs[0][2]}' has {hogs[0][0]} open connections — possible bandwidth hog!")
        tip(f"Consider closing or limiting: {hogs[0][2]}")

# ─────────────────────────────────────────────
# 4. DNS BENCHMARK
# ─────────────────────────────────────────────
def dns_benchmark():
    header("🔎 DNS Response Time Benchmark")

    dns_servers = {
        "Google (8.8.8.8)":         "8.8.8.8",
        "Cloudflare (1.1.1.1)":     "1.1.1.1",
        "OpenDNS (208.67.222.222)": "208.67.222.222",
        "Quad9 (9.9.9.9)":         "9.9.9.9",
    }

    results = []
    test_domain = "www.google.com"

    for name, ip in dns_servers.items():
        try:
            start = time.time()
            if OS == "Windows":
                cmd = ["nslookup", test_domain, ip]
            else:
                cmd = ["dig", f"@{ip}", test_domain, "+time=2", "+tries=1"]
            subprocess.run(cmd, capture_output=True, timeout=4)
            elapsed = (time.time() - start) * 1000
            results.append((elapsed, name, ip))
            color = Fore.GREEN if elapsed < 50 else Fore.YELLOW if elapsed < 150 else Fore.RED
            print(f"  {name:<30} {color}{elapsed:.1f} ms")
        except Exception:
            bad(f"{name:<30} Timed out")

    if results:
        results.sort()
        best_name, best_ip = results[0][1], results[0][2]
        ok(f"Fastest DNS: {best_name}")
        tip(f"Set your DNS to {best_ip} for faster browsing.\n"
            f"     (Router → DNS Settings → Primary DNS → {best_ip})")

# ─────────────────────────────────────────────
# 5. FLUSH DNS CACHE
# ─────────────────────────────────────────────
def flush_dns():
    header("🧹 Flush DNS Cache")
    try:
        if OS == "Windows":
            subprocess.run(["ipconfig", "/flushdns"], capture_output=True)
        elif OS == "Darwin":
            subprocess.run(["sudo", "dscacheutil", "-flushcache"], capture_output=True)
        else:
            subprocess.run(["sudo", "systemd-resolve", "--flush-caches"],
                           capture_output=True)
        ok("DNS cache flushed successfully.")
    except Exception as e:
        warn(f"Could not flush DNS: {e}")

# ─────────────────────────────────────────────
# 6. PING GATEWAY (ROUTER CHECK)
# ─────────────────────────────────────────────
def ping_gateway():
    header("🏠 Router / Gateway Ping Test")
    gateways = psutil.net_if_stats()

    # Try to find default gateway
    gateway_ip = None
    try:
        if OS == "Windows":
            result = subprocess.run(["ipconfig"], capture_output=True, text=True)
            for line in result.stdout.splitlines():
                if "Default Gateway" in line and line.strip().split()[-1] != "":
                    gateway_ip = line.strip().split()[-1]
                    break
        else:
            result = subprocess.run(["ip", "route"], capture_output=True, text=True)
            for line in result.stdout.splitlines():
                if line.startswith("default"):
                    gateway_ip = line.split()[2]
                    break
    except Exception:
        pass

    if not gateway_ip or gateway_ip == "0.0.0.0":
        gateway_ip = "192.168.1.1"  # common default

    info(f"Testing gateway: {gateway_ip}")

    ping_cmd = ["ping", "-n", "5", gateway_ip] if OS == "Windows" \
               else ["ping", "-c", "5", gateway_ip]
    try:
        result = subprocess.run(ping_cmd, capture_output=True, text=True, timeout=15)
        output = result.stdout

        # Extract avg ping
        if OS == "Windows":
            for line in output.splitlines():
                if "Average" in line:
                    print(f"  {Fore.CYAN}{line.strip()}")
        else:
            for line in output.splitlines():
                if "avg" in line or "rtt" in line:
                    print(f"  {Fore.CYAN}{line.strip()}")

        if "unreachable" in output.lower() or "100%" in output:
            bad("Cannot reach the router! Check cable/Wi-Fi connection.")
        else:
            ok(f"Router is reachable at {gateway_ip}")

    except Exception as e:
        bad(f"Ping failed: {e}")

# ─────────────────────────────────────────────
# 7. LIVE BANDWIDTH MONITOR
# ─────────────────────────────────────────────
def live_monitor(duration=10):
    header(f"📊 Live Bandwidth Monitor ({duration} seconds)")
    print(f"  {'Time':<8} {'Download':<20} {'Upload':<20}")
    print(f"  {'─'*45}")

    prev = psutil.net_io_counters()
    for i in range(duration):
        time.sleep(1)
        curr = psutil.net_io_counters()
        dl = mb(curr.bytes_recv - prev.bytes_recv)
        ul = mb(curr.bytes_sent - prev.bytes_sent)
        color = Fore.RED if dl > 5 else Fore.GREEN
        print(f"  {i+1:<8} {color}↓ {dl:.3f} MB/s{Style.RESET_ALL}         "
              f"{Fore.BLUE}↑ {ul:.3f} MB/s")
        prev = curr

# ─────────────────────────────────────────────
# 8. RECOMMENDATIONS SUMMARY
# ─────────────────────────────────────────────
def print_recommendations(d_mbps):
    header("💡 Recommendations Summary")

    recommendations = [
        ("Move closer to the router",
         "5GHz signal weakens quickly with distance and walls."),
        ("Change Wi-Fi channel to 36 or 149",
         "Avoids interference with neighbors on same channel."),
        ("Set DNS to 1.1.1.1 (Cloudflare)",
         "Faster DNS = faster page loads and connection setup."),
        ("Restart router every week",
         "Clears memory leaks and refreshes connections."),
        ("Close background apps",
         "Updates, cloud sync, and streaming hog your bandwidth silently."),
        ("Use Ethernet cable when possible",
         "Always faster and more stable than any Wi-Fi band."),
        ("Check with ISP for throttling",
         "Some ISPs slow speeds after daily usage thresholds, not just monthly quota."),
    ]

    if d_mbps and d_mbps < 10:
        recommendations.insert(0, ("Call your ISP immediately",
                                   f"Your speed ({d_mbps} Mbps) is far below acceptable range."))

    for i, (rec, reason) in enumerate(recommendations, 1):
        print(f"\n  {Fore.YELLOW}{i}. {rec}")
        print(f"     {Fore.WHITE}{reason}")

# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────
def main():
    print(f"\n{Fore.CYAN}{'█'*55}")
    print(f"{'█'*3}  {Fore.YELLOW}Network Speed Diagnostic & Optimizer Tool  {Fore.CYAN}{'█'*3}")
    print(f"{'█'*55}{Style.RESET_ALL}")
    print(f"  OS: {platform.system()} {platform.release()}")
    print(f"  Time: {time.strftime('%Y-%m-%d %H:%M:%S')}")

    check_interfaces()
    ping_gateway()
    find_bandwidth_hogs()
    dns_benchmark()
    flush_dns()
    live_monitor(duration=10)

    # Speed test last (heaviest operation)
    d_mbps, u_mbps, ping = run_speed_test()

    print_recommendations(d_mbps)

    header("✅ Diagnosis Complete")
    if d_mbps:
        print(f"  Final Result: Download = {Fore.GREEN}{d_mbps} Mbps  "
              f"{Fore.WHITE}| Upload = {Fore.BLUE}{u_mbps} Mbps  "
              f"{Fore.WHITE}| Ping = {Fore.YELLOW}{ping:.1f} ms")
    print(f"\n  {Fore.CYAN}Run this script again after applying fixes to compare results.\n")

if __name__ == "__main__":
    main()

[*] Installing speedtest-cli...

███████████████████████████████████████████████████████
███  Network Speed Diagnostic & Optimizer Tool  ███
███████████████████████████████████████████████████████
  OS: Windows 10
  Time: 2026-05-12 23:27:44

═══════════════════════════════════════════════════════
  🔌 Network Interfaces
═══════════════════════════════════════════════════════
  🌐 Loopback Pseudo-Interface 1 IP: 127.0.0.1          Link: 1073 Mbps
  📶 Wi-Fi           IP: 10.176.158.202     Link: 72 Mbps

═══════════════════════════════════════════════════════
  🏠 Router / Gateway Ping Test
═══════════════════════════════════════════════════════
  [i] Testing gateway: 10.176.158.12
  Minimum = 2ms, Maximum = 3ms, Average = 2ms
  [✓] Router is reachable at 10.176.158.12

═══════════════════════════════════════════════════════
  🐷 Bandwidth Hog Processes (Top 10)
═══════════════════════════════════════════════════════
  Sampling for 3 seconds...

  Total in last 3 sec → ↓ 0.67 MB  |  ↑ 0.22 